# 第 4 周练习 — Stayez Python 到 C++ 算法优化器

**学生：** Vagz1216  
**课程：** LLM Engineering — Andela AI Engineering Bootcamp  
**练习：** 第 4 周 — AI 辅助代码工程（Python → C++）

---

## 练习目标（理念）

把第 4 周的 **AI 代码工程**用到 Stayez 场景：目前定价在 WooCommerce 里偏静态；规模上来后需要**动态 surge 定价**（周末、竞争、热度）。

在 Python 里对大量并发搜索做定价会很慢。本工具扮演 AI 驱动的「高级 C++ 工程师」：把慢速 Python 定价逻辑转成高性能 C++，目标是每秒处理海量计算。

## 和本课概念的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| 多模型路由 | Groq / Hugging Face / Gemini 同一套 messages |
| system prompt | 强制「只回原始 C++」 |
| Gradio 并排面板 | 左侧 Python，右侧生成的 C++ |
| 基准算法 | 百万次定价循环作为「慢代码」样例 |

## 怎么跑

1. `.env` 准备：`GROQ_API_KEY`、`GEMINI_API_KEY`、`HF_TOKEN`
2. 从上到下运行单元格，最后 `demo.launch(inbrowser=True)`
3. 可先点 **Run Python** 看基线输出，再选模型点 **Convert to C++**

### 特点

- 引擎：**Hugging Face (Llama 3.2 3B)**、**Groq (Llama 3.3 70B)**、**Gemini 1.5 Flash**
- Gradio 并排代码面板（语法高亮）
- 内置贴近真实的 Stayez 动态定价模拟算法


In [ ]:
# ========== 导入：环境、I/O 捕获、Gradio、多厂商客户端 ==========

# os：读环境变量里的各家 API Key / Token
import os
# io：用 StringIO 截获 exec 时的 stdout
import io
# sys：临时替换 sys.stdout，便于捕获 print
import sys
# gradio：搭并排代码转换 UI
import gradio as gr
# load_dotenv：加载 .env
from dotenv import load_dotenv
# OpenAI 兼容客户端：给 Groq / Gemini 用
from openai import OpenAI
# Hugging Face InferenceClient：走 HF 推理 API
from huggingface_hub import InferenceClient


In [ ]:
# ========== API 密钥与多客户端：HF / Groq / Gemini + 模型注册表 ==========

# 加载 .env（覆盖已有同名变量）
load_dotenv(override=True)

# 分别读取三家密钥（变量名与 env 键保持原样）
groq_api_key   = os.getenv('GROQ_API_KEY')
gemini_api_key = os.getenv('GEMINI_API_KEY')
hf_token       = os.getenv('HF_TOKEN')

# 1) Hugging Face 推理客户端（开源模型经 HF API）
hf_client = InferenceClient(token=hf_token)

# 2) Groq：OpenAI 兼容 base_url，适合大模型低延迟推理
groq = OpenAI(api_key=groq_api_key, base_url="https://api.groq.com/openai/v1")

# 3) Gemini：同样用 OpenAI 兼容封装指向 Google 端点
gemini = OpenAI(api_key=gemini_api_key, base_url="https://generativelanguage.googleapis.com/v1beta/openai/")

# 下拉显示名 -> (model_id, client)；id 与 URL 相关字符串保持原样
MODELS = {
    "Groq (Llama 3.3 70B)": ("llama-3.3-70b-versatile", groq),
    "Hugging Face (Llama 3.2 3B)": ("meta-llama/Llama-3.2-3B-Instruct", hf_client),
    "Gemini 1.5 Flash": ("gemini-1.5-flash", gemini),
}

# 启动自检提示（emoji 与英文保持原样）
print("✅ Clients ready: Hugging Face, Groq, Gemini")


In [ ]:
# ========== system prompt：约束模型只输出可编译的高性能 C++ ==========

# 三引号多行字符串：发给模型的系统指令（英文 STRICT RULES 保持原样）
system_prompt = """\
You are a Senior C++ Software Engineer working on performance-critical backend systems.
Your task is to translate Python code into high-performance, idiomatic C++ code.

STRICT RULES:
1. Respond ONLY with raw C++ code. No markdown code fences, no explanations.
2. Include necessary #include headers (<iostream>, <cmath>, <iomanip>, etc) and a main() function.
3. Use appropriate data types like double for precision and int64_t for large loops.
4. Prioritize maximum runtime performance.
5. Produce identical output to the original Python code, formatting numbers consistently.
"""

# 把用户粘贴的 Python 包进 user 消息（前缀英文保持原样）
def build_user_prompt(python_code):
    return f"Convert this Python code to high-performance C++:\n\n{python_code}"


In [ ]:
# ========== 转换函数 + 本地跑 Python：供 Gradio 两个按钮调用 ==========

def port_to_cpp(model_choice, python_code):
    # 空输入：返回 C++ 注释形式的提示（英文保持原样）
    if not python_code.strip():
        return "// Please paste some Python code to translate."

    # 从注册表取出真实 model_id 与对应 client
    model_id, client = MODELS[model_choice]
    # 标准 Chat messages：system 定规则，user 放待转换代码
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user",   "content": build_user_prompt(python_code)}
    ]

    try:
        # 低 temperature 更稳定；max_tokens 给足长代码
        response = client.chat.completions.create(
            model=model_id,
            messages=messages,
            temperature=0.1,
            max_tokens=4000
        )
        # 取出助手回复并去首尾空白
        reply = response.choices[0].message.content.strip()
        # 逐个剥掉模型可能加的 Markdown 围栏标记
        for fence in ["```cpp", "```c++", "```c", "```"]:
            reply = reply.replace(fence, "")
        return reply.strip()

    except Exception as e:
        # 失败时仍返回可显示在 C++ 面板里的注释文本
        return f"// Translation error:\n// {e}"


def run_python(python_code):
    """Execute the Python code and capture its stdout output."""
    # 内存缓冲区：承接 print 内容
    buf = io.StringIO()
    # 备份真正的 stdout，finally 里恢复
    old_stdout = sys.stdout
    sys.stdout = buf
    try:
        # 在受限 globals 里 exec；保留 builtins 以便 print 等可用
        exec(python_code, {"__builtins__": __builtins__})
        return buf.getvalue()
    except Exception as e:
        # 运行错误文案前缀保持英文
        return f"Error: {e}"
    finally:
        # 无论成败都恢复 stdout，避免弄坏笔记本后续输出
        sys.stdout = old_stdout


In [ ]:
# ========== 预置基准：Stayez 动态定价百万次循环（作为左侧默认源码）==========

# 整段字符串会填进 Gradio Code 组件；内部可执行逻辑与 print 文案保持原样
python_example = """# Stayez Dynamic Pricing Benchmark 
# 模拟 1,000,000 个并发访客配置的定价计算
# Simulates pricing calculations for 1,000,000 concurrent guest configurations
import time
import math

def calculate_stayez_price(base_ksh, check_in_days_from_now, nights, 
                           competing_properties, reviews_count):
    total = 0.0
    for night in range(nights):
        day_price = base_ksh
        current_day = check_in_days_from_now + night
        
        # 周末激增：+30%（周五/周六/周日）
        # Weekend surge: +30% (Friday/Saturday/Sunday)
        if current_day % 7 >= 4:
            day_price *= 1.30
            
        # 稀缺因素：竞争对手越少=收费越高
        # Scarcity factor: fewer competitors = charge more
        scarcity = math.exp(-competing_properties / 10.0)
        day_price *= (1 + 0.2 * scarcity)
        
        # 人气溢价：更多评论=可信=价格更高
        # Popularity premium: more reviews = trusted = priced higher
        popularity_premium = math.log1p(reviews_count) * 0.01
        day_price *= (1 + popularity_premium)
        
        total += day_price
        
    return total

def run_stayez_pricing_benchmark(iterations):
    grand_total_revenue = 0.0
    
    # 模拟计算一百万种不同预订场景的价格
    # Simulate calculating prices for a million different booking scenarios
    for i in range(iterations):
        days_from_now = i % 30
        nights = (i % 5) + 1
        competing = (i % 20)
        reviews = i % 100
        
        price = calculate_stayez_price(5000, days_from_now, nights, competing, reviews)
        grand_total_revenue += price
        
    return round(grand_total_revenue, 2)

start = time.time()
total_projected_revenue = run_stayez_pricing_benchmark(1_000_000)
end = time.time()

print(f"Total Projected Revenue: KSh {total_projected_revenue:,.2f}")
print(f"Calculation took {end - start:.4f} seconds in Python")
"""


In [ ]:
# ========== Gradio UI：选模型、左侧 Python / 右侧 C++、绑定两个按钮 ==========

# Soft 主题 + 自定义标题
with gr.Blocks(theme=gr.themes.Soft(), title="Stayez Python-to-C++ Optimizer") as demo:
    # 标题与简短说明（界面英文保持原样）
    gr.Markdown("## Stayez Algorithm Optimizer — Python to C++")
    gr.Markdown(
        "Use frontier AI models to translate slow Python pricing algorithms into high-performance C++ code.  \n"
        "Select a model, then click **Convert to C++**."
    )

    # 第一行：模型下拉 + 转换按钮
    with gr.Row():
        model_dd = gr.Dropdown(
            choices=list(MODELS.keys()),
            value="Groq (Llama 3.3 70B)",
            label="AI Code Engineer",
            scale=3
        )
        convert_btn = gr.Button("Convert to C++ →", variant="primary", scale=1)

    # 第二行：左右两列代码面板
    with gr.Row():
        with gr.Column():
            # 左侧：可编辑 Python 源码，默认填入基准算法
            python_input = gr.Code(
                label="Python Code (Source)",
                value=python_example,
                language="python",
                lines=35
            )
            # 本地执行 Python，看 stdout
            run_py_btn = gr.Button("▶ Run Python")
            python_out = gr.Textbox(label="Python Output", lines=3, interactive=False)

        with gr.Column():
            # 右侧：展示模型生成的 C++
            cpp_output = gr.Code(
                label="C++ Code (Generated)",
                value="// Click 'Convert to C++' to generate...",
                language="cpp",
                lines=35
            )

    # 绑定：转换走 port_to_cpp；运行走 run_python
    convert_btn.click(fn=port_to_cpp, inputs=[model_dd, python_input], outputs=[cpp_output])
    run_py_btn.click(fn=run_python, inputs=[python_input], outputs=[python_out])

# inbrowser=True：启动后尝试自动打开浏览器
demo.launch(inbrowser=True)
